# Simulated Annealing for the N-Queens Problem

## Author Details
- **Author:** Ramya Mercy Rajan  
- **Course:** MSc Software Engineering  
- **University:** University of Europe for Applied Sciences  
- **Professor:** Raja Hashim Ali  

---

# Algorithm Overview

Simulated Annealing (SA) is a local search algorithm inspired by
the process of cooling hot metal slowly to reach a stable state.

For the N-Queens problem, the algorithm starts with a random
board arrangement and tries to reduce the number of queen conflicts
step by step.

During each iteration:

1. A random queen is selected.
2. The queen is moved to a different random column.
3. The change in conflicts is calculated.
4. If the move improves the solution, it is accepted.
5. If the move increases conflicts, it may still be accepted
   based on probability.

This helps the algorithm escape from local minima and continue
searching for better solutions.

As the temperature decreases, the algorithm becomes more greedy
and accepts fewer worse moves.

---

# Optimization Used

A normal implementation recalculates all queen conflicts after
every move, which is slow for larger board sizes.

To improve performance, only the conflicts related to the moved
queen are recalculated instead of checking the whole board again.

This reduced the computation time significantly, especially for
larger values of \(N\).

---

# Cooling Schedule

The following parameters were used in this project:

- Initial temperature: \(T_0 = 100\)
- Cooling factor: \(\alpha = 0.995\)
- Maximum iterations: 60,000

After every iteration, the temperature is reduced gradually until
it becomes very close to zero.

---

# Time Complexity

The overall time complexity of the algorithm is:

\[
O(\text{max\_steps} \times N)
\]

because only one queen is checked during each iteration instead
of comparing all queen pairs.

In [6]:
import time
import random
import math
import psutil
import os

In [7]:
def measure_memory():
    """Return current process RSS memory in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

In [8]:
def row_conflicts_count(board, r):
    """
    Count how many queens attack the queen in row r.
    Scans all other rows: O(N).
    """
    N = len(board)
    c = board[r]
    count = 0
    for rr in range(N):
        if rr == r:
            continue
        cc = board[rr]
        if cc == c or abs(cc - c) == abs(rr - r):
            count += 1
    return count

In [9]:
def total_conflicts_count(board):
    """
    Count all conflicting queen pairs: O(N²).
    Used only for the initial conflict computation.
    """
    N = len(board)
    count = 0
    for i in range(N):
        for j in range(i + 1, N):
            ci, cj = board[i], board[j]
            if ci == cj or abs(ci - cj) == abs(i - j):
                count += 1
    return count

In [10]:
def simulated_annealing(N, max_steps=60_000, temp=100.0, cooling=0.995):
    start      = time.time()
    mem_before = measure_memory()

    
    board        = [random.randint(0, N - 1) for _ in range(N)]
    current_conf = total_conflicts_count(board)   # O(N²) once

    
    for step in range(max_steps):

        
        if current_conf == 0:
            break

        
        r     = random.randint(0, N - 1)
        new_c = random.randint(0, N - 1)

        if new_c == board[r]:
            temp *= cooling     
            continue

        
        old_row_conf = row_conflicts_count(board, r)   

        old_c    = board[r]
        board[r] = new_c
        new_row_conf = row_conflicts_count(board, r)   

        
        delta = new_row_conf - old_row_conf

        
        if delta <= 0:
            
            current_conf += delta
        elif random.random() < math.exp(-delta / max(temp, 1e-10)):
            
            current_conf += delta
        else:
            
            board[r] = old_c

        
        temp *= cooling

    elapsed   = time.time() - start
    mem_after = measure_memory()

    return {
        "solution":        board if current_conf == 0 else None,
        "final_conflicts": current_conf,
        "time_sec":        round(elapsed, 4),
        "memory_MB":       round(mem_after - mem_before, 4),
    }



In [11]:
def print_board(board):
    """Print the chessboard with Q for queens and . for empty squares."""
    N = len(board)
    print()
    for row in range(N):
        line = ""
        for col in range(N):
            line += " Q " if board[row] == col else " . "
        print(line)
    print()

In [12]:
def run_experiments():
    test_sizes = [10, 30, 50, 100, 200, 500]

    print("=" * 60)
    print("  N-Queens — Simulated Annealing")
    print("=" * 60)
    print(f"  T₀={100}, α={0.995}, max_steps={60_000}")

    for N in test_sizes:
        print(f"\n>>> N = {N}")
        result = simulated_annealing(N)

        solved = result["solution"] is not None
        print(f"  Solved           : {solved}")
        print(f"  Final conflicts  : {result['final_conflicts']}")
        print(f"  Time             : {result['time_sec']} s")
        print(f"  Memory delta     : {result['memory_MB']} MB")

        if solved:
            print(f"  Solution         : {result['solution']}")
            if N <= 10:
                print_board(result["solution"])
        else:
            print(f"  Result           : {result['final_conflicts']} conflicts "
                  f"remain at termination.")


if __name__ == "__main__":
    run_experiments()

  N-Queens — Simulated Annealing
  T₀=100, α=0.995, max_steps=60000

>>> N = 10
  Solved           : True
  Final conflicts  : 0
  Time             : 0.046 s
  Memory delta     : 0.1328 MB
  Solution         : [4, 2, 5, 9, 6, 3, 0, 7, 1, 8]

 .  .  .  .  Q  .  .  .  .  . 
 .  .  Q  .  .  .  .  .  .  . 
 .  .  .  .  .  Q  .  .  .  . 
 .  .  .  .  .  .  .  .  .  Q 
 .  .  .  .  .  .  Q  .  .  . 
 .  .  .  Q  .  .  .  .  .  . 
 Q  .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  Q  .  . 
 .  Q  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  Q  . 


>>> N = 30
  Solved           : True
  Final conflicts  : 0
  Time             : 0.1431 s
  Memory delta     : 0.0273 MB
  Solution         : [4, 29, 22, 18, 15, 21, 0, 16, 26, 7, 5, 11, 24, 1, 13, 20, 2, 23, 25, 27, 12, 8, 6, 19, 17, 14, 28, 10, 3, 9]

>>> N = 50
  Solved           : True
  Final conflicts  : 0
  Time             : 0.243 s
  Memory delta     : 0.0352 MB
  Solution         : [48, 15, 7, 26, 23, 0, 36, 31, 28, 19, 1, 11, 27, 4